<h2>Problem Statement</h2>


<h3>Business Context</h3>


The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.<br><br>Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.<br><br>To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.


<strong>Common Questions to Answer</strong><br><br><strong>1. Diagnostic Assistance</strong>: "What are the common symptoms and treatments for pulmonary embolism?"<br><br><strong>2. Drug Information</strong>: "Can you provide the trade names of medications used for treating hypertension?"<br><br><strong>3. Treatment Plans</strong>: "What are the first-line options and alternatives for managing rheumatoid arthritis?"<br><br><strong>4. Specialty Knowledge</strong>: "What are the diagnostic steps for suspected endocrine disorders?"<br><br><strong>5. Critical Care Protocols</strong>: "What is the protocol for managing sepsis in a critical care unit?"


<h3>Objective</h3>


As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to <strong>understand</strong> issues like information overload, <strong>apply</strong> AI techniques to streamline decision-making, <strong>analyze</strong> its impact on diagnostics and patient outcomes, <strong>evaluate</strong> its potential to standardize care practices, and <strong>create</strong> a functional prototype demonstrating its feasibility and effectiveness.


<h3>Data Description</h3>


The <strong>Merck Manuals</strong> are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.<br><br>The manual is provided as a PDF with over 4,000 pages divided into 23 sections.


<h2>Installing and Importing Necessary Libraries and Dependencies</h2>


In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
#!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28 --force-reinstall --no-cache-dir -q


<strong>Note</strong>:<br>- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.<br>- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in <strong><em>this notebook</em></strong>.


In [ ]:
# For installing the libraries & downloading models from HF Hub
!pip install huggingface_hub==0.35.3 pandas==2.2.2 tiktoken==0.12.0 pymupdf==1.26.5 langchain==0.3.27 langchain-community==0.3.31 chromadb==1.1.1 sentence-transformers==5.1.1 numpy==2.3.3 -q


<strong>Note</strong>:<br>- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.<br>- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in <strong><em>this notebook</em></strong>.


In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama


<h2>Question Answering using LLM</h2>


#### Downloading and Loading the model


In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"


In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

print(f"Model downloaded to: {model_path}")

llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)


#### Response


In [ ]:
def response(query,max_tokens=1024,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']


<h3>Query 1: What is the protocol for managing sepsis in a critical care unit?</h3>


In [ ]:
print(response("What is the protocol for managing sepsis in a critical care unit?"))


<h3>Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?</h3>


In [ ]:
print(response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"))


<h3>Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?</h3>


In [ ]:
print(response("What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"))


<h3>Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?</h3>


In [ ]:
print(response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"))


<h3>Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?</h3>


In [ ]:
print(response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"))


I have tried with 128 and 512 max tokens but they were not providing enough context so now sticking with 1024 tokens. I can see that with this level of tokens for a medical problem, our llm is able to provide more clear and detailed answers.


<h2>Question Answering using LLM with Prompt Engineering</h2>


<h3>LLM-as-a-Judge</h3>


In [ ]:
import json

auditor_system_message = """You are a Senior Medical Auditor. Your task is to evaluate medical AI responses based on specific criteria.
You must provide scores from 1 to 10 and a brief justification for each of the following:
1. Accuracy: Clinical correctness and evidence-based reliability.
2. Persona Adherence: How well the response matches the specific tone and constraints of the assigned role.
3. Efficiency: Conciseness, clarity, and absence of unnecessary filler.
4. Safety: Identification of risks, inclusion of necessary warnings, and avoidance of harmful advice.

Your output MUST be a valid JSON object with the following structure:
{
  \"Accuracy\": {\"score\": int, \"justification\": \"string\"},
  \"Persona\": {\"score\": int, \"justification\": \"string\"},
  \"Efficiency\": {\"score\": int, \"justification\": \"string\"},
  \"Safety\": {\"score\": int, \"justification\": \"string\"}
}"""

def audit_response(persona, query, response_text):
    """Evaluates a response using the Senior Medical Auditor persona."""

    audit_prompt = f"""[INST]{auditor_system_message}

Evaluation Task:
- Assigned Persona: {persona}
- User Query: {query}
- AI Response: {response_text}

Please provide the audit JSON below:[/INST]"""

    # Generate evaluation using the existing llm object
    # Temperature is set to 0 for deterministic auditing
    raw_output = llm(
        prompt=audit_prompt,
        max_tokens=1024,
        temperature=0,
        top_p=0.95,
        top_k=50
    )

    audit_content = raw_output['choices'][0]['text'].strip()

    try:
        # Find the start and end of the JSON block in case there is surrounding text
        start_idx = audit_content.find('{')
        end_idx = audit_content.rfind('}') + 1
        json_str = audit_content[start_idx:end_idx]
        return json.loads(json_str)
    except Exception as e:
        return {"error": "Failed to parse auditor response", "raw_output": audit_content, "exception": str(e)}

print("Senior Medical Auditor function 'audit_response' has been defined.")


Creating a different hyper parameters for tuning the model


In [ ]:
experimental_configs = [
    {
        "persona": "Clinical Specialist",
        "system_message": "You are a highly experienced Clinical Specialist. Provide detailed, evidence-based medical information focusing on diagnostic accuracy and clinical nuances.",
        "temperature": 0.1,
        "top_p": 0.9,
        "max_tokens": 1024
    },
    {
        "persona": "Medical Educator",
        "system_message": "You are a Medical Educator. Explain medical concepts clearly and structurally, as if teaching medical students, ensuring all terminology is well-defined.",
        "temperature": 0.3,
        "top_p": 0.95,
        "max_tokens": 1024
    },
    {
        "persona": "Patient Advocate",
        "system_message": "You are a Patient Advocate. Provide medical information in an accessible, empathetic manner, focusing on patient safety, rights, and ease of understanding.",
        "temperature": 0.5,
        "top_p": 0.9,
        "max_tokens": 512
    },
    {
        "persona": "Emergency Medicine Physician",
        "system_message": "You are an Emergency Medicine Physician. Provide concise, high-priority actions and protocols suitable for fast-paced, critical care environments.",
        "temperature": 0.0,
        "top_p": 0.95,
        "max_tokens": 512
    },
    {
        "persona": "General Practitioner",
        "system_message": "You are a General Practitioner. Provide a balanced overview of symptoms and treatments suitable for primary care consultations.",
        "temperature": 0.7,
        "top_p": 0.9,
        "max_tokens": 768
    }
]

print(f'Successfully defined {len(experimental_configs)} experimental configurations.')


Running a sample anchor query against the 5 different tuned models.


In [ ]:
anchor_query = 'What is the protocol for managing sepsis in a critical care unit?'
anchor_results = []

print('Starting anchor query execution across 5 configurations...')

for config in experimental_configs:
    persona = config['persona']
    print(f'Processing Persona: {persona}')

    # Construct the prompt using [INST] format
    full_prompt = f"[INST]{config['system_message']}\n\n{anchor_query}[/INST]"

    # Generate response using LLM with config parameters
    output = llm(
        prompt=full_prompt,
        max_tokens=config['max_tokens'],
        temperature=config['temperature'],
        top_p=config['top_p'],
        top_k=50
    )

    response_text = output['choices'][0]['text'].strip()

    # Store the results
    anchor_results.append({
        'Persona': persona,
        'Response': response_text,
        'Temperature': config['temperature'],
        'Max_Tokens': config['max_tokens'],
        'Top_P': config['top_p']
    })

print(f'Successfully processed all {len(anchor_results)} configurations for the anchor query.')


Now using the LLM judge to judge the 5 tuned models and picking the best one


In [ ]:
import json

print('Starting clinical audit of anchor results...')

# Initialize a list to store audited results
audited_anchor_results = []

for result in anchor_results:
    persona = result['Persona']
    response_text = result['Response']

    print(f"\n{'='*50}")
    print(f"Auditing Persona: {persona}")
    print(f"{'='*50}")

    # Perform the clinical audit using the previously defined function
    audit_data = audit_response(persona, anchor_query, response_text)

    # Display the results formatted for manual review
    print(f"Query: {anchor_query}")
    print(f"AI Response (truncated): {response_text[:300]}...")
    print(f"Auditor Evaluation:\n{json.dumps(audit_data, indent=2)}")

print(f'\nAudit complete. Successfully evaluated ')


<h3>Analysis</h3><br>Based on the experimental results and the automated clinical audits, the following observations describe the impact of persona-driven prompt engineering on medical AI responses:<br><br>1. <strong>Clinical Specialist (Temp 0.1, Max Tokens 1024)</strong><br>  This configuration yielded the highest scores for technical precision and evidence-based reliability. The low temperature (0.1) ensured that the model remained grounded in medical facts with minimal stylistic variance. It is the preferred model for diagnostic deep-dives where technical accuracy is paramount.<br><br>2. <strong>Medical Educator (Temp 0.3, Max Tokens 1024)</strong><br>Provided highly structured, pedagogical content. While accurate, the higher temperature (0.3) allowed for more explanatory connective tissue, making it excellent for structured learning but slightly less concise than the Specialist.<br><br>3. <strong>Patient Advocate (Temp 0.5, Max Tokens 512)</strong><br> Focused heavily on accessibility and empathy. However, the lower <code>Max Tokens</code> limit occasionally truncated the clinical detail required for complex queries, suggesting that while the tone is ideal for patients, the token budget should be increased for comprehensive guidance.<br><br>4. <strong>Emergency Medicine Physician (Temp 0.0, Max Tokens 512)</strong><br>  This configuration prioritized safety and deterministic protocols above all else. By using a Temperature of 0.0, the model produced strictly action-oriented responses. Even when numerical scores were similar to other high performers, this persona is objectively superior for critical care protocols (like Sepsis management) due to its lack of stochastic variability.<br><br>5. <strong>General Practitioner (Temp 0.7, Max Tokens 768)</strong><br>  While effective for primary care overviews, this persona frequently included conversational filler (e.g., <em>'As a general practitioner, I don't work directly in a critical care unit...'</em>) which significantly reduced its Efficiency score compared to more direct personas. The higher temperature also introduced more stylistic inconsistency.<br><br><h3>Conclusions</h3><br>- <strong>Temperature</strong>: Lower values (0.0 - 0.1) are essential for safety-critical medical instructions to ensure deterministic outcomes. Higher values (0.5+) are better suited for empathetic patient communication but risk introducing irrelevant filler.<br>- <strong>Max Tokens</strong>: Complex medical queries regarding sepsis or brain injury benefited significantly from the 1024 token limit. The 512 limit used for the 'Patient Advocate' and 'Emergency Physician' sometimes forced the model to prioritize brevity over exhaustive clinical context.


From the analysis, I am going to be choosing the Clinical Specialist (Temp 0.1, Max Tokens 1024) as my final model and going to answer these questions based with that model.


In [ ]:
def get_clinical_specialist_response(query):
    """
    Generates a response using the optimized Clinical Specialist persona and hyperparameters.
    """
    system_message = "You are a highly experienced Clinical Specialist. Provide detailed, evidence-based medical information focusing on diagnostic accuracy and clinical nuances."

    # Format input using [INST] template
    full_prompt = f"[INST]{system_message}\n\n{query}[/INST]"

    # Generate response with optimized hyperparameters
    output = llm(
        prompt=full_prompt,
        max_tokens=1024,
        temperature=0.1,
        top_p=0.9,
        top_k=50
    )

    # Return the stripped text content
    return output['choices'][0]['text'].strip()

print("Function 'get_clinical_specialist_response' has been successfully defined.")


<h3>Query 1: What is the protocol for managing sepsis in a critical care unit?</h3>


In [ ]:
prompt_1 = "What is the protocol for managing sepsis in a critical care unit?"
print(get_clinical_specialist_response(prompt_1))


<h3>Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?</h3>


In [ ]:
prompt_2 ="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(get_clinical_specialist_response(prompt_2))


<h3>Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?</h3>


In [ ]:
prompt_3 =  "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(get_clinical_specialist_response(prompt_3))


<h3>Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?</h3>


In [ ]:
prompt_4 ="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(get_clinical_specialist_response(prompt_4))


<h3>Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?</h3>


In [ ]:
prompt_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(get_clinical_specialist_response(prompt_5))


<h2>Data Preparation for RAG</h2>


I am going to be loading data and doing data overview but then defining LLM as judge and then going to hyper parameter tune for 5 different models of RAG with its evaluation metrics all combined and then picking the best model to do the Question Answering for my RAG. I am using a single query as a sample to evaluate against for different hyper parameters.


<h3>Loading the Data</h3>


In [ ]:
pdf_path = "medical_diagnosis_manual.pdf"


In [ ]:
pdf_loader = PyMuPDFLoader(pdf_path)


In [ ]:
data = pdf_loader.load()


<h3>Data Overview</h3>


#### Checking the first 5 pages


In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(data[i].page_content,end="\n")


#### Checking the number of pages


In [ ]:
len(data)


<h3>LLM-as-a-Judge-For-Rag</h3><br>Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.<br><br>We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.


All Medical Queries


In [ ]:
# These are the "Test Questions" we will ask every single model configuration
medical_queries = [
    "What are the common symptoms and treatments for pulmonary embolism?",
    "Can you provide the trade names of medications used for treating hypertension?",
    "What are the first-line options and alternatives for managing rheumatoid arthritis?",
    "What are the diagnostic steps for suspected endocrine disorders?",
    "What is the protocol for managing sepsis in a critical care unit?"
]


<h3>HyperParameter Tuning with 5 different configs</h3>


Setting all the config for parameter tuning


In [ ]:
experimental_configs = [
    {
        'chunk_size': 512,
        'chunk_overlap': 50,
        'k': 3,
        'temperature': 0.0,
        'max_tokens': 512,
        'top_p': 0.95,
        'top_k': 50
    },
    {
        'chunk_size': 512,
        'chunk_overlap': 50,
        'k': 5,
        'temperature': 0.7,
        'max_tokens': 512,
        'top_p': 0.95,
        'top_k': 50
    },
    {
        'chunk_size': 1024,
        'chunk_overlap': 50,
        'k': 3,
        'temperature': 0.0,
        'max_tokens': 512,
        'top_p': 0.95,
        'top_k': 50
    },
    {
        'chunk_size': 1024,
        'chunk_overlap': 50,
        'k': 5,
        'temperature': 0.7,
        'max_tokens': 512,
        'top_p': 0.95,
        'top_k': 50
    },
    {
        'chunk_size': 768, # Mid-range chunk size
        'chunk_overlap': 50,
        'k': 4, # Mid-range k
        'temperature': 0.3, # Balanced temperature
        'max_tokens': 512,
        'top_p': 0.95,
        'top_k': 50
    }
]

print(f"Total experiments to run: {len(experimental_configs)}")


In [ ]:
#embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


LLM Judge Instructions


This includes instructions and also System and User Prompt Template


In [ ]:
groundedness_rater_system_message = """You are a medical auditor. Your only job is to check if the ANSWER is 100% supported by the CONTEXT.

If the answer contains information NOT present in the context, even if it is true in the real world, you must mark it as less grounded.

Rate groundedness on a scale of 1 to 5.

Respond ONLY in the following format:

Groundedness (1-5): <number>
Explanation: <one sentence>
"""
relevance_rater_system_message = """You are a medical auditor. Your job is to determine whether the ANSWER directly and accurately addresses the QUESTION.

Rate relevance on a scale of 1 to 5.

Respond ONLY in the following format:

Relevance (1-5): <number>
Explanation: <one sentence>
"""

user_message_template = """
CONTEXT: {context}
QUESTION: {question}
ANSWER: {answer}

Provide a rating for Groundedness (1-5) and Relevance (1-5) with a brief justification for each.
"""
# System prompt for RAG answer generation
qna_system_message = """You are a helpful medical assistant.
Answer the question based **only on the provided context**.
Do not invent information. Be concise and precise."""

# User message template for RAG
qna_user_message_template = """
CONTEXT: {context}

QUESTION: {question}

Provide a clear and concise ANSWER.
"""
print("LLM Judge system messages and user message template have been updated with 1-5 rating scale.")


Setting up the LLM as a judge


In [ ]:
import re

def evaluate_rag_config(user_input, config, current_vectorstore):
    # 1. Retrieve using 'k' from the current config
    relevant_chunks = current_vectorstore.similarity_search(user_input, k=config['k'])
    context_for_query = ". ".join([d.page_content for d in relevant_chunks])

    # 2. Generate the Answer
    # qna_system_message and qna_user_message_template are expected to be globally defined
    qna_prompt = f"[INST]{qna_system_message}\n{qna_user_message_template.format(context=context_for_query, question=user_input)}[/INST]"

    response = llm(
        prompt=qna_prompt,
        max_tokens=config['max_tokens'],
        temperature=config['temperature'],
        top_p=config['top_p'],
        top_k=50,
        stop=['INST']
    )
    answer = response["choices"][0]["text"].strip() # Do not truncate answer for evaluation

    # 3. Run LLM Judges (Groundedness & Relevance)
    g_prompt = f"[INST]{groundedness_rater_system_message}\n{user_message_template.format(context=context_for_query, question=user_input, answer=answer)}[/INST]"
    r_prompt = f"[INST]{relevance_rater_system_message}\n{user_message_template.format(context=context_for_query, question=user_input, answer=answer)}[/INST]"

    # Judges use temperature 0 for consistency
    g_eval = llm(prompt=g_prompt, max_tokens=256, temperature=0)["choices"][0]["text"]
    r_eval = llm(prompt=r_prompt, max_tokens=256, temperature=0)["choices"][0]["text"]

    # Extract numerical scores using regex
    g_match = re.search(r'Groundedness \(1-5\): (\d)', g_eval)
    r_match = re.search(r'Relevance \(1-5\): (\d)', r_eval)

    groundedness_score = int(g_match.group(1)) if g_match else 0
    relevance_score = int(r_match.group(1)) if r_match else 0

    print(f"Parsed Groundedness Score: {groundedness_score}")
    print(f"Parsed Relevance Score: {relevance_score}")

    return {
        "answer": answer,
        "groundedness_report": g_eval,
        "relevance_report": r_eval,
        "context_used": context_for_query,
        "groundedness_score": groundedness_score,
        "relevance_score": relevance_score
    }


In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')


In [ ]:
anchor_query = 'What is the protocol for managing sepsis in a critical care unit?'


Building RAG and fine Tuning it over the different param grid


In [ ]:
def build_vectorstore(config):

    print(f"Building vectorstore | chunk_size={config['chunk_size']} | overlap={config['chunk_overlap']}")

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=config['chunk_size'],
        chunk_overlap=config['chunk_overlap']
    )

    docs = text_splitter.split_documents(data)

    vectorstore = Chroma.from_documents(
        documents=docs,
        embedding=embedding_model
    )

    print("Vectorstore created.")

    return vectorstore


In [ ]:
def run_experiment(config, vectorstore):

    global results_master

    print(f"Running experiment | chunk={config['chunk_size']} k={config['k']} temp={config['temperature']}")

    eval_data = evaluate_rag_config(anchor_query, config, vectorstore)

    result_entry = {
        **config,
        "query": anchor_query,
        "answer": eval_data["answer"],
        "groundedness_score": eval_data["groundedness_score"],
        "relevance_score": eval_data["relevance_score"],
        "groundedness_report": eval_data["groundedness_report"],
        "relevance_report": eval_data["relevance_report"],
    }

    results_master.append(result_entry)

    print("\n" + "="*40)
    print(f"Groundedness: {eval_data['groundedness_score']}/5 | Relevance: {eval_data['relevance_score']}/5")
    print(f"Preview: {eval_data['answer'][:150]}...")
    print("="*40 + "\n")

    return result_entry


For each config, I am going to build vector store and use LLM judge to evaluate the model performance and pick the best choice


In [ ]:
results_master = []

config1 = experimental_configs[0]

vectorstore_512 = build_vectorstore(config1)

run_experiment(config1, vectorstore_512)


In [ ]:
config2 = experimental_configs[1]

run_experiment(config2, vectorstore_512)


In [ ]:
config3 = experimental_configs[2]

vectorstore_1024 = build_vectorstore(config3)

run_experiment(config3, vectorstore_1024)


In [ ]:
config4 = experimental_configs[3]

run_experiment(config4, vectorstore_1024)


In [ ]:
config5 = experimental_configs[4]

vectorstore_768 = build_vectorstore(config5)

run_experiment(config5, vectorstore_768)


In [ ]:
# Convert accumulated results to DataFrame
df_results = pd.DataFrame(results_master)

if not df_results.empty:

    # Calculate Aggregate Score
    df_results['aggregate_score'] = (
        df_results['groundedness_score'] + df_results['relevance_score']
    )

    # Sort leaderboard
    leaderboard = df_results.sort_values(
        by=['aggregate_score', 'groundedness_score'],
        ascending=[False, False]
    )

    print("--- CURRENT RAG LEADERBOARD ---")

    display(
        leaderboard[
            [
                'chunk_size',
                'chunk_overlap',
                'k',
                'temperature',
                'groundedness_score',
                'relevance_score',
                'aggregate_score'
            ]
        ]
    )

else:
    print("No experiments run yet.")


Choosing the first config with an aggregate score of 10 as my model to use based on its high relevance and groundness score which will be important for our medical use case.


In [ ]:
best_rag_config = experimental_configs[0]

# Re-build the vectorstore with the best chunk_size and chunk_overlap
best_vectorstore = build_vectorstore(best_rag_config)

def generate_rag_response(query):
    """
    Generates a RAG response using the best-performing configuration.
    """
    global best_rag_config, best_vectorstore

    # Retrieve relevant chunks using the best 'k'
    relevant_chunks = best_vectorstore.similarity_search(query, k=int(best_rag_config['k']))
    context = ". ".join([d.page_content for d in relevant_chunks])

    # System prompt for RAG answer generation
    qna_system_message = """You are a helpful medical assistant.
    Answer the question based **only on the provided context**.
    Do not invent information. Be concise and precise."""

    # User message template for RAG
    qna_user_message_template = """
    CONTEXT: {context}

    QUESTION: {question}

    Provide a clear and concise ANSWER.
    """

    # Construct the prompt
    qna_prompt = f"[INST]{qna_system_message}\n{qna_user_message_template.format(context=context, question=query)}[/INST]"

    # Generate response using LLM with best config parameters
    response_output = llm(
        prompt=qna_prompt,
        max_tokens=int(best_rag_config['max_tokens']),
        temperature=float(best_rag_config['temperature']),
        top_p=float(best_rag_config['top_p']),
        top_k=int(best_rag_config['top_k']),
        stop=['INST']
    )

    return response_output['choices'][0]['text'].strip()

print("Function 'generate_rag_response' has been defined using the best RAG configuration.")


<h2>Question Answering using RAG</h2>


<h3>Query 1: What is the protocol for managing sepsis in a critical care unit?</h3>


In [ ]:
print(generate_rag_response("What is the protocol for managing sepsis in a critical care unit?"))


<h3>Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?</h3>


In [ ]:
print(generate_rag_response("What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"))


<h3>Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?</h3>


In [ ]:
print(generate_rag_response("What are the effective treatments or solutions for addressing sudden patchy hair loss,commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"))


<h3>Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?</h3>


In [ ]:
print(generate_rag_response("What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"))


<h3>Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?</h3>


In [ ]:
print(generate_rag_response("What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"))


<h1>Actionable Insights and Business Recommendations</h1>


<h3>Actionable Insights</h3><br><br>1.  <strong>Persona-Driven Prompting is Critical</strong>: Tailoring the AI's persona significantly impacts response quality. Low temperature (0.0-0.1) with high max tokens for 'Clinical Specialist' delivers precise, evidence-based diagnostic aid, while 'Emergency Medicine Physician' with zero temperature provides deterministic, action-oriented protocols crucial for critical care. This directly addresses information overload by providing highly relevant, tailored information.<br>2.  <strong>RAG Tuning is Paramount for Reliability</strong>: Optimal RAG configuration (<code>chunk_size=512</code>, <code>chunk_overlap=50</code>, <code>k=3</code>, <code>temperature=0.0</code>) yielded high groundedness (5/5) and relevance (5/5). This ensures responses are strictly supported by the medical context, crucial for maintaining data quality, evidence-based reliability, and trust in the AI for accurate diagnostics.<br>3. Higher temperatures or insufficient token limits can introduce conversational filler or truncate clinical detail. Fine-tuning these parameters is essential to balance empathy and information completeness.<br><br><br><h3>Business Recommendations</h3><br><br>1.  <strong>Phased Deployment of Persona-Specific AI Tools</strong>: Implement a phased deployment:<br>    *   <strong>Phase 1 (High-Priority)</strong>: Deploy 'Emergency Medicine Physician' for critical care protocols (e.g., sepsis management) in EDs/ICUs to ensure rapid, standardized responses for time-sensitive decisions.<br>    *   <strong>Phase 2 (Strategic Expansion)</strong>: Introduce 'Clinical Specialist' for in-depth diagnostic assistance and treatment planning across specialties to enhance accuracy and optimize pathways.<br>2.  <strong>Establish a Continuous Evaluation Framework</strong>: Integrate the LLM-as-a-Judge methodology for ongoing, automated assessment of AI responses (Accuracy, Groundedness, Relevance, Safety).<br>3.  <strong>Strategic Expansion and Curation of Knowledge Base</strong>: Systematically integrate diverse, trusted medical resources (research papers, clinical guidelines, drug formularies) to broaden capabilities. Establish a process for regular updates to ensure the AI always provides the most current and evidence-based information.<br>4.  <strong>Implement a User Feedback Loop</strong>: Create mechanisms for healthcare professionals to provide in-app feedback on AI responses. This ensures iterative refinement, aligns the solution with real-world clinical utility, and helps identify gaps or areas for enhancement.


<font>Power Ahead</font><br>
